# cleanlab.ipynb

## Gemini prompt

 I am using a random forest model as a binary classifier for coconut palm tree silhouettes. My training set contains 6 Hu moments as features and "alive" or "dead" as labels. Please provide a Jupyter notebook which shows me how to use Python CleanLab for curating my dataset.

 ## Gemini response

 To audit and curate a binary "alive" / "dead" coconut palm silhouette dataset using CleanLab and a Random Forest classifier, the critical step is computing out-of-sample predicted probabilities (pred_probs) via cross-validation (cross_val_predict) rather than in-sample predict_proba (which is overfit and makes CleanLab miss true label errors).   
 
 Below is a complete, runnable Jupyter notebook structure using Datalab (for label issues, outliers, and near-duplicates) as well as CleanLearning for noise-robust Random Forest training.    

In [1]:
import numpy as np
import pandas as pd
from cleanlab import Datalab
from cleanlab.classification import CleanLearning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from icecream import ic
import duckdb
from IPython.display import HTML, display
import joblib

In [2]:
# Inputs

# created by pca.ipynb
# ['tree_id', 'dead']
shape_classes_csv = 'tree_silhouettes_202611201700/shape_class.csv' 

# created by pca.ipynb; Hu moments scaled with "np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-15)"
# ['tree_id', 'h1', ..., 'h7]
hu_dataset_csv = 'hu_dataset_202611201700.csv' 

# created by                     
training_dataset_csv = 'training_dataset_202611221400.csv'        

## Cell 1: Data preparation

In [3]:
df = pd.read_csv(training_dataset_csv)
df

,tree_id,dead,h1,h2,h3,h4,h5,h6,h7
0,77,0,-0.524697,-1.482314,-2.328288,-2.914501,-5.633616,-3.723588,-5.756312
1,179,0,-0.058715,-0.151640,-0.348927,-0.381639,-0.746923,-0.458197,3.379063
2,198,0,0.008442,-0.017242,-0.533576,-0.607849,-1.178735,-0.620762,-2.727663
3,214,0,-0.242159,-0.589162,-1.049723,-1.081010,-2.146467,-1.393966,3.835318
4,218,0,0.278500,0.551319,-0.444942,-0.489195,-0.956292,-0.223162,-2.899226
...,...,...,...,...,...,...,...,...,...
495,26698,0,-0.348136,-0.820270,-1.167873,-1.270130,-2.489414,-1.687492,3.931697
496,26880,0,0.227771,0.445403,-0.206446,-0.225895,-0.442066,-0.003300,3.189195
497,26887,0,-0.248995,-0.593392,-0.592517,-0.668806,-1.299471,-0.966028,3.666268
498,26948,0,-0.674271,-2.903272,-3.169987,-4.325097,9.990137,-5.816504,-8.072671


In [4]:
# parse dataset into numpy arrays
X = df.drop(['tree_id', 'dead', 'h7'], axis=1).to_numpy()
ic(X)
y = df[['dead']].to_numpy().flatten()
ic(y)
meta = df[['tree_id']].to_numpy().flatten()
ic(meta);

ic| X: array([[-5.24697335e-01, -1.48231437e+00, -2.32828812e+00,
               -2.91450121e+00, -5.63361568e+00, -3.72358808e+00],
              [-5.87151096e-02, -1.51640099e-01, -3.48927187e-01,
               -3.81638680e-01, -7.46922795e-01, -4.58196644e-01],
              [ 8.44151371e-03, -1.72421804e-02, -5.33576316e-01,
               -6.07848806e-01, -1.17873464e+00, -6.20762163e-01],
              ...,
              [-2.48994679e-01, -5.93392474e-01, -5.92516953e-01,
               -6.68805734e-01, -1.29947109e+00, -9.66027548e-01],
              [-6.74271045e-01, -2.90327154e+00, -3.16998725e+00,
               -4.32509723e+00,  9.99013708e+00, -5.81650391e+00],
              [ 4.80641042e-01,  9.38626599e-01,  1.43389476e+00,
                1.40301237e+00,  2.82146174e+00,  1.87212970e+00]],
             shape=(500, 6))
ic| y: array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,

## Cell 2: Out-of-Sample Predicted Probabilities (pred_probs)

CleanLab's Confident Learning requires out-of-sample pred_probs of shape (N, K) where K = 2. Using StratifiedKFold preserves the "alive" / "dead" class ratio across folds.

In [5]:
model_path = 'rf_tree_shape_model20261124T0233Z.joblib'
loaded_model = joblib.load(model_path)
# predictions = loaded_model.predict(X_test)   


In [6]:
# rf_base = RandomForestClassifier(
#     n_estimators=200,
#     max_depth=None,
#     min_samples_leaf=2,
#     class_weight="balanced",
#     n_jobs=-1,
#     random_state=42,
# )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-sample probabilities for every training sample
pred_probs = cross_val_predict(
    # rf_base,
    loaded_model,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)

print("pred_probs shape:", pred_probs.shape)
print("Out-of-sample ROC AUC:", roc_auc_score(y, pred_probs[:, 1]))

pred_probs shape: (500, 2)
Out-of-sample ROC AUC: 0.9407744874715263


## Cell 3: Audit Dataset with Datalab

Datalab flags suspected label errors and can also surface outliers or near-duplicate silhouettes that distort tree splits.

In [7]:
# find issues

df = df.reset_index(drop=True) # make sure index is 0, 1, 2, ...
lab = Datalab(data=df, label_name="dead", task="classification")
lab.find_issues(pred_probs=pred_probs)
lab.report()

Finding label issues ...
Finding outlier issues ...
Fitting OOD estimator based on provided pred_probs ...
Finding non_iid issues ...
Finding class_imbalance issues ...

Audit complete. 51 issues found in the dataset.
Dataset Information: num_examples: 500, num_classes: 2

Here is a summary of various issues found in your data:

issue_type  num_issues
     label          30
   outlier          20
   non_iid           1

Learn about each issue: https://docs.cleanlab.ai/stable/cleanlab/datalab/guide/issue_type_description.html
See which examples in your dataset exhibit each issue via: `datalab.get_issues(<ISSUE_NAME>)`

Data indices corresponding to top examples of each issue are shown below.


----------------------- label issues -----------------------

About this issue:
	Examples whose given label is estimated to be potentially incorrect
    (e.g. due to annotation error) are flagged as having label issues.
    

Number of examples with this issue: 30
Overall dataset quality in terms 

In [8]:
# add issues to original dataset
df_issues = lab.get_issues()
df_with_issues = df.join(df_issues)
# df_with_issues.rename(columns={'y_proba': 'machine', 'y_test': 'human', 'meta_test': 'tree_id'}, inplace=True)
df_with_issues

,tree_id,dead,h1,h2,h3,h4,h5,h6,h7,is_label_issue,label_score,is_outlier_issue,outlier_score,is_non_iid_issue,non_iid_score,is_class_imbalance_issue,class_imbalance_score
0,77,0,-0.524697,-1.482314,-2.328288,-2.914501,-5.633616,-3.723588,-5.756312,False,1.000000,False,0.315899,False,0.558816,False,1.0
1,179,0,-0.058715,-0.151640,-0.348927,-0.381639,-0.746923,-0.458197,3.379063,False,0.982000,False,0.284978,False,0.971869,False,1.0
2,198,0,0.008442,-0.017242,-0.533576,-0.607849,-1.178735,-0.620762,-2.727663,False,0.984405,False,0.288992,False,0.910571,False,1.0
3,214,0,-0.242159,-0.589162,-1.049723,-1.081010,-2.146467,-1.393966,3.835318,False,0.968333,False,0.262830,False,0.927127,False,1.0
4,218,0,0.278500,0.551319,-0.444942,-0.489195,-0.956292,-0.223162,-2.899226,False,0.800869,False,0.070693,False,0.953728,False,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,26698,0,-0.348136,-0.820270,-1.167873,-1.270130,-2.489414,-1.687492,3.931697,False,1.000000,False,0.315899,False,0.552192,False,1.0
496,26880,0,0.227771,0.445403,-0.206446,-0.225895,-0.442066,-0.003300,3.189195,False,0.946944,False,0.230354,False,0.878407,False,1.0
497,26887,0,-0.248995,-0.593392,-0.592517,-0.668806,-1.299471,-0.966028,3.666268,False,0.991667,False,0.301330,False,0.834029,False,1.0
498,26948,0,-0.674271,-2.903272,-3.169987,-4.325097,9.990137,-5.816504,-8.072671,False,1.000000,False,0.315899,False,0.554467,False,1.0


#### Images best displayed using a light theme (Ctrl K T)

In [9]:
for row in duckdb.sql('SELECT tree_id, dead, label_score FROM df_with_issues WHERE is_label_issue AND dead=1 ORDER BY label_score').fetchall():
   tree_id, dead, label_score = row
   print(f'tree_id: {tree_id}\n  dead: {dead}\nlabel_score: {label_score: .2f}\n')
   display(HTML(f'<img src="tree_silhouettes_202611201700/{tree_id}.svg" width="200">')) 

tree_id: 5546
  dead: 1
label_score:  0.00



tree_id: 16661
  dead: 1
label_score:  0.01



tree_id: 21158
  dead: 1
label_score:  0.03



tree_id: 26054
  dead: 1
label_score:  0.08



In [10]:
for row in duckdb.sql('SELECT tree_id, dead, label_score FROM df_with_issues WHERE is_label_issue AND dead=0 ORDER BY label_score').fetchall():
   tree_id, dead, label_score = row
   print(f'tree_id: {tree_id}\n  dead: {dead}\nlabel_score: {label_score: .2f}\n')
   display(HTML(f'<img src="tree_silhouettes_202611201700/{tree_id}.svg" width="200">')) 

tree_id: 21131
  dead: 0
label_score:  0.01



tree_id: 18814
  dead: 0
label_score:  0.03



tree_id: 18396
  dead: 0
label_score:  0.05



tree_id: 3088
  dead: 0
label_score:  0.06



tree_id: 5513
  dead: 0
label_score:  0.07



tree_id: 3404
  dead: 0
label_score:  0.07



... (16 outputs left out)

tree_id: 9169
  dead: 0
label_score:  0.25



tree_id: 15456
  dead: 0
label_score:  0.26



tree_id: 5717
  dead: 0
label_score:  0.33



tree_id: 6375
  dead: 0
label_score:  0.34



## Cell 4: Rank & Inspect Suspected Label Errors

Sort suspected issues by label_quality_score (lower score = higher likelihood of a mislabeled silhouette) and map integer predictions back to "alive" / "dead".

In [11]:
label_issues = lab.get_issues("label")

flagged = (
    label_issues.assign(
        given_label=[int_to_label[i] for i in y],
        predicted_label=[int_to_label[i] for i in pred_probs.argmax(axis=1)],
        prob_alive=pred_probs[:, 1],
        prob_dead=pred_probs[:, 0],
    )
    .query("is_label_issue")
    .sort_values("label_quality_score", ascending=True)
)

print(f"Flagged {len(flagged)} potential label errors out of {len(df)} samples.")
flagged[["given_label", "predicted_label", "label_quality_score", "prob_alive", "prob_dead"]].head(15)

NameError: name 'int_to_label' is not defined

## Cell 5: Curate Dataset & Train a Noise-Robust Random Forest

You have two complementary paths after reviewing flagged silhouettes:

1. Manual / Threshold Curation: Inspect low-score rows (especially silhouettes with ambiguous crowns or motion blur), relabel clear mistakes, and prune unresolvable outliers.

2. CleanLearning Wrapper: Automatically filters out estimated noisy labels across cross-validation folds before fitting the final Random Forest.

In [ ]:
# Option A: Drop rows flagged as label issues (after manual review in production)
clean_mask = ~label_issues["is_label_issue"].to_numpy()
X_clean = X.loc[clean_mask].reset_index(drop=True)
y_clean = y[clean_mask]

rf_curated = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
rf_curated.fit(X_clean, y_clean)

# Option B: CleanLearning wrapper on original (X, y) using the same CV splitter
cl_rf = CleanLearning(clf=rf_base, cv_n_folds=5)
cl_rf.fit(X, y)

# Inspect label issues estimated by CleanLearning
cl_issues = cl_rf.get_label_issues()
print("CleanLearning estimated label errors:", int(cl_issues["is_label_issue"].sum()))

## Cell 6: Evaluate Curated Model on a Held-Out Test Set

Always evaluate on an uncleaned, held-out test set (X_test, y_test) so you measure real generalization rather than artificial accuracy gains from removing hard examples from evaluation.

In [ ]:
# Assuming you split X_test, y_test before curation:
# y_pred_test = rf_curated.predict(X_test)
# print(classification_report(y_test, y_pred_test, target_names=["dead", "alive"]))

## Practical Notes for Hu Moments & Palm Silhouettes

- Scale of Hu Moments: Raw Hu moments span many orders of magnitude and can be near zero or negative; apply a signed log transform (np.sign(h) * np.log10(np.abs(h) + 1e-12)) before training if you haven't already.

- Silhouette Ambiguity: Dead palms that still retain upright fronds or have partially snapped trunks often sit near decision boundaries—treat low label_quality_score rows as priority visual review candidates before bulk-dropping them.

#### Cleanlab in Python: Find Label Errors Before Training Your Model (YouTube)   

This walkthrough video demonstrates how to combine scikit-learn and CleanLab to catch noisy labels before model training so bad annotations do not skew your classifier.   

Want to add code to inspect and export flagged rows for visual review?

Ways to extend this curation workflow:

- Add signed log-transform preprocessing for Hu moments
- Audit outliers and near-duplicates alongside label errors